In [2]:
import numpy as np
import pandas as pd
import os
import sys
import glob
import tqdm
import re
from typing import List

os.chdir("/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis")
sys.path.append("../../benchmark")
import test_base
from sentence_splitter import split_text_into_sentences
from BERT_classifier.Classify_report_with_BERT import classification_report_BERT
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [3]:
sentence_length = 6

## Test using a BERT-Classification Model trained on paragraphs to classify an entire report.

**Function:** pdf -> NACE Class

In [4]:
dataset_path = "data/datasets/german_annual_reports"
dataset_path = "data/datasets/stoxx_600_extended"
dataset_path = "data/datasets/reports_subset_from_full_data_1"
dataset_path = "data/datasets/stoxx_600"

In [5]:
over_view_df_path = os.path.join(dataset_path, os.path.basename(dataset_path) + "_overview.csv")

dataset_path_texts = os.path.join(dataset_path, "TXTs")

dataset_name = os.path.basename(dataset_path)

In [6]:
%pwd

'/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis'

In [7]:
nace_classes = pd.read_csv(over_view_df_path, index_col=0, sep=",")
nace_classes.head()

,Name,Symbol,FactSet ID,Revenue - 2022 (in EUR),Revenue - 2023 (in EUR),Revenue - 2024 (in EUR),NACE,NACE_letter,Report
0,SalMar ASA,SALM-NO,SALM-NO,1984.965581,2458.824022,2271.61166349053,3.21,A,salmar-annual-report-2022.pdf
1,Bakkafrost P/F,BAKKA-NO,BAKKA-NO,929.503079,937.862170,973.053513491168,3.21,A,Bakkafrost PF2.pdf
2,Antofagasta plc,ANTO-GB,ANTO-GB,5577.681426,5849.975673,6113.94698310345,7.29,B,Antofagasta plc1.pdf
3,Anglo American plc,AAL-GB,AAL-GB,33423.271144,28355.894415,25288.1884924262,7.29,B,Anglo American plc1.pdf
4,TotalEnergies SE,TTE-FR,TTE-FR,250538.948328,202517.658053,180837.266896225,6.10,B,Totalenergies EP Gabon1.pdf


In [8]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class = {report[0][:-4] + ".txt": report[1] for report in report_to_nace_class.items()}
len(report_to_nace_class)

294

In [9]:
reports_path = glob.glob(os.path.join(dataset_path_texts, "*.txt"))
len(reports_path)

263

In [10]:
def get_tables(lines: list): 
    tables = []
    current_table = []

    for line in lines:
        if line.strip().startswith("|"):  # line belongs to a table
            current_table.append(line.strip())
        else:
            if current_table:  # table ended
                tables.append("\n".join(current_table))
                current_table = []

    # catch last table if file ends without empty lines
    if current_table:
        tables.append("\n".join(current_table))

    return tables

def preprocess_report(pdf_path: str) -> List[str]:

    with open(pdf_path, "r") as f: 
        text = f.read()
    
    lines = text.split("\n")

    tables = get_tables(lines)

    # drop if condidtion is True
    conditions = [
        # filter images
        lambda line: line == '<!-- image -->',
        
        #filter tables 
        lambda line: (line[0] == "|" and line[-1] == "|") if len(line) > 1 else False, 

        # filter headers
        lambda line: line.strip()[0] == "#" if len(line) > 0 else True,

        # filter sentences
        lambda line: "." not in line,
        
        # more than 50% is numbers
        lambda line: sum(ch.isalpha() for ch in line) / len(line) < 0.5,

        # minimum 3 words 
        lambda line: len(re.sub(r"[^a-zA-ZäöüÄÖÜß\s]", '', line).strip().split(" ")) < 3,

        # Minimum 2 Sentences
        #lambda line: sum([0 if len(sentence.split(" ")) < 3 else 1 for sentence in split_text_into_sentences(line, "en")]) < 2

    ]
    accepted_lines = [line for line in lines if not any(condition(line) for condition in conditions)]
    accepted_lines += tables

    chunks = []

    for line in accepted_lines: 
        sentences = split_text_into_sentences(line, language='en')
        sentences = [sentence.strip() for sentence in sentences]
        sentences = [sentence for sentence in sentences if sentence != ""]
        new_chunks = [(" ".join(sentences[i:i+sentence_length])).strip() for i in range(0, len(sentences), 3)]

        chunks += new_chunks

    # if there is only one sentence in the last chunk, balance the two last chunks
    if len(split_text_into_sentences(chunks[-1], language = "en")) == 1: 
        last_two_chunks = chunks[-2] + " " + chunks[-1]
        chunks[-2] = last_two_chunks[0:(len(last_two_chunks) + 1) // 2]
        chunks[-1] = last_two_chunks[(len(last_two_chunks) + 1) // 2: (len(last_two_chunks)) - (len(last_two_chunks) + 1) // 2]

    chunks = [re.sub(r'\b\d+\.\d+\b', '', chunk) for chunk in chunks]
    chunks = [re.sub(r"[^a-zA-ZäöüÄÖÜß.\s]", '', chunk) for chunk in chunks]
    chunks = [re.sub(r"\s+", " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'\.{2,}', " ", chunk) for chunk in chunks]
    chunks = [re.sub(r'^\d+\.\s*', " ", chunk) for chunk in chunks]
    chunks = [chunk.lower() for chunk in chunks]
    chunks = [chunk.strip() for chunk in chunks]

    return chunks

In [11]:
%ls results/BERT_models/

finbert__train_full_model__all_labels/
results__all_labels__train_classifier_only/
results__bert-base-uncased__train_full_model__all_labels/
results__bert-base-uncased__train_full_model__some_labels/
results__cos_thresh_045bert-base-uncased__train_full_model__some_labels/
results__cos_thresh_045finbert__train_full_model__all_labels/
results__cos_thresh_05bert-base-uncased__train_full_model__some_labels/
results__cos_thresh_05finbert__train_full_model__all_labels/
results_data_2__cos_thresh_045bert-base-uncased__train_full_model__some_labels/
results_data_2__cos_thresh_045finbert__train_full_model__all_labels/
results_data_2__cos_thresh_05bert-base-uncased__train_full_model__some_labels/
results_data_2__cos_thresh_05finbert__train_full_model__all_labels/
results_data_2__cos_thresh_06bert-base-uncased__train_full_model__some_labels/
results__finbert__train_full_model__all_labels/
results__finbert__train_full_model__some_labels/
results__new_approach_data__num_layers_2bert-base-uncased__t

In [12]:
ckpt = "results/BERT_models/results_null_classifiers__cos_thres_0.5__bert-base-uncased__train_full_model__some_labels/checkpoint-2331"
ckpt = "results/BERT_models/results__new_approach_data__num_layers_2bert-base-uncased__train_full_model__some_labels/checkpoint-15990"
model = classification_report_BERT.load_custom_bert_from_checkpoint(ckpt_path=ckpt)
tokenizer = AutoTokenizer.from_pretrained(ckpt) 


In [13]:
reports_path

['data/datasets/stoxx_600/TXTs/Hannover Rueck SE1.txt',
 'data/datasets/stoxx_600/TXTs/Interpump Group S.p.A.1.txt',
 'data/datasets/stoxx_600/TXTs/Intertek Group PLC1.txt',
 'data/datasets/stoxx_600/TXTs/Anheuser-Busch InBev SANV3.txt',
 'data/datasets/stoxx_600/TXTs/Scout24 SE3.txt',
 'data/datasets/stoxx_600/TXTs/Bridgepoint Group Plc1.txt',
 'data/datasets/stoxx_600/TXTs/Financiere de Tubize SA2.txt',
 'data/datasets/stoxx_600/TXTs/Severn Trent Plc1.txt',
 'data/datasets/stoxx_600/TXTs/Bakkafrost PF2.txt',
 'data/datasets/stoxx_600/TXTs/Ferrari NV2.txt',
 'data/datasets/stoxx_600/TXTs/Sika AG3.txt',
 'data/datasets/stoxx_600/TXTs/Swiss Prime Site AG2.txt',
 'data/datasets/stoxx_600/TXTs/Poste Italiane SpA2.txt',
 'data/datasets/stoxx_600/TXTs/Haleon PLC1.txt',
 'data/datasets/stoxx_600/TXTs/Land Securities Group PLC2.txt',
 'data/datasets/stoxx_600/TXTs/Rentokil Initial plc2.txt',
 'data/datasets/stoxx_600/TXTs/Nexans SA3.txt',
 'data/datasets/stoxx_600/TXTs/SEB SA2.txt',
 'data/da

In [ ]:
for i in range(1,2):
    nace_level = i

    result_path = f"results/BERT_classification/2_dataset__{dataset_name}_sentence_len_{sentence_length}__nace_level_{nace_level}"

    res = test_base.test_report_classification(
        reports_path=reports_path,
        preprocess_report=preprocess_report, 
        report_to_nace_class=report_to_nace_class, 
        result_path = result_path,
        level=i,
        overwrite=False, 
        classification_function=classification_report_BERT.classify_report, 
        path_nace_code_descriptions="data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", 
        model=model,
        tokenizer=tokenizer,)

  0%|                                                                                                                                                                                                                                                   | 0/263 [00:00<?, ?it/s]

['chairman of the board of directors hannover re bermuda ltd. hamilton bermuda', 'member of the board of directors hannover life reassurance company of america bermuda ltd. hamilton bermuda', 'chairman of the board of directors argenta holdings limited london united kingdom chairman of the board of directors glencar insurance company orlando usa chairman of the board of directors hannover re services usa inc. illinois usa member of the board of directors argenta syndicate management limited london united kingdom member of the board of directors hdi global specialty se hannover germany', 'chairman of the board of directors hannover retakaful b.s.c. c manama bahrain deputy chairman of the board of directors hannover life re of australasia ltd sydney australia deputy chairman of the board of directors hannover reinsurance group africa pty ltd. johannesburg south africa deputy chairman of the board of directors hannover re south africa limited johannesburg south africa member of the board 

  0%|▉                                                                                                                                                                                                                                          | 1/263 [00:10<44:09, 10.11s/it]

{'F': -1.3616739511489868, 'C': -3.249000310897827, 'NO_CLASS': 4.5098958015441895, 'M': -0.7763649225234985, 'L': -0.839199423789978, 'H': -0.8372454643249512, 'P': -2.181816339492798, 'G': 0.5814843773841858, 'D': -1.524138331413269, 'J': -3.4036147594451904, 'Q': -1.2192769050598145, 'I': -0.5815463066101074, 'N': 1.319612979888916, 'E': -1.6537365913391113, 'K': 4.1230340003967285, 'A': -2.4251790046691895, 'B': -2.247896671295166}
['dividends refer to the year of formation of the distributed profit.', 'a inclusive of the debt related to the acquisition of investments.', 'after two difficult years due to the covid pandemic the markets were supported by the relaxation of restrictions in most economies. this contributed to an upturn in consumption with particularly good results during the first part of . major help also came from governmentand eufunded support and relaunch measures including those set out in respectively the national recovery and resilience plan pnrr and the nextgene

  1%|█▊                                                                                                                                                                                                                                       | 2/263 [00:34<1:19:42, 18.32s/it]

...

{'F': -0.5555026531219482, 'C': -1.1919580698013306, 'NO_CLASS': 4.282467365264893, 'M': -1.6640291213989258, 'L': -1.967132329940796, 'H': -0.4953902065753937, 'P': -1.2100573778152466, 'G': 2.35617995262146, 'D': 0.06742171943187714, 'J': -3.2541942596435547, 'Q': 0.22755777835845947, 'I': -1.3614137172698975, 'N': -0.3980370759963989, 'E': -1.7416316270828247, 'K': 0.5895543694496155, 'A': -2.457453727722168, 'B': -2.2330615520477295}
['an explanation of the alternative performance measures apms used by the group including underlying profit before tax underlying ebitda and reported and underlying pro forma earnings per share is set out on pages to along with a reconciliation to statutory measures.', 'bridgepoint is an international alternative asset fund management group with offices in europe the us and china. we support growth businesses with a european focus and seek to create value by helping to build companies with greatly enhanced longterm potential.', 'each of which has been 

  2%|█████▎                                                                                                                                                                                                                                   | 6/263 [02:36<2:16:39, 31.90s/it]

{'F': -1.464989185333252, 'C': -3.5215296745300293, 'NO_CLASS': 4.58701753616333, 'M': -0.7778115272521973, 'L': -0.6527760624885559, 'H': -0.7811943292617798, 'P': -2.518786907196045, 'G': 0.2924724519252777, 'D': -1.7283778190612793, 'J': -3.5762925148010254, 'Q': -1.5842218399047852, 'I': -0.3307233154773712, 'N': 1.7797746658325195, 'E': -1.580448865890503, 'K': 4.675706386566162, 'A': -2.4595441818237305, 'B': -2.246084213256836}
['the board of directors of financire de tubize has established the annual report. this report is available on the website www.financieretubize.be', 'profit for the financial year million million in', 'increase of outstanding bank borrowings from million at december to million at december', 'acquisition of ucb shares increasing the holding of the company in ucb from on december to on december . if the general shareholders meeting of april approves the annual accounts including the prop', '']


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 40.98it/s]


{'F': -1.4003123044967651, 'C': -3.750308036804199, 'NO_CLASS': 5.39563512802124, 'M': -1.252322793006897, 'L': -1.0490468740463257, 'H': -0.8517027497291565, 'P': -2.1217005252838135, 'G': 1.1047331094741821, 'D': -1.636160135269165, 'J': -3.9372360706329346, 'Q': -0.46480220556259155, 'I': -0.8560917377471924, 'N': 1.339683175086975, 'E': -2.1225435733795166, 'K': 4.493409156799316, 'A': -2.789735794067383, 'B': -2.5972747802734375}
